# HypatiaX — Complete Reproducible Pipeline

**Paper:** *HypatiaX: A Hybrid Symbolic-Neural Framework for Extrapolation-Reliable Analytical Discovery*  
**Author:** Ruperto Pedro Bonet Chaple  
**Venue:** JMLR v3.0, April 2026

This notebook reproduces all paper results:
- **Exp 1:** DeFi 74-task benchmark (§10.2–10.4, §10.6)
- **Exp 1b:** Portfolio Variance seed sweep (§10.5)
- **Exp 2:** Feynman 30-equation extrapolation (§10.7)
- **Exp 3:** Nguyen-12 SR suite (§10.8)
- **Supp A:** Hybrid routing improvements
- **Supp B:** Noise & sample-complexity sweep
- **§10.9:** Instability analysis (K=30)

> **Run order:** Execute cells top to bottom. Full run takes 16-32 hours on Kaggle 4-vCPU.

In [ ]:
# Cell 1: Platform Setup & Environment
import os
import sys
import json
import random
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# ── Paper-quality: FAST=0 ────────────────────────────────────────────────
os.environ['FAST'] = '0'

# ── Seeds (DO NOT CHANGE for paper reproducibility) ──────────────────────
os.environ.setdefault('NN_SEED', '42')
os.environ.setdefault('PYSR_SEED', '42')
os.environ.setdefault('PYTHONHASHSEED', '42')

# ── Platform detection ────────────────────────────────────────────────────
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    print('Platform: Google Colab')
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except: pass
elif IS_KAGGLE:
    print('Platform: Kaggle (12-hour limit)')
else:
    print('Platform: Local')

print(f'FAST={os.environ["FAST"]} (paper-quality)')
print(f'Seeds: NN={os.environ["NN_SEED"]} / PYSR={os.environ["PYSR_SEED"]}')

# ── Paths ─────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd()
RESULTS_DIR = REPO_ROOT / "hypatiax" / "data" / "results"
LOG_DIR = REPO_ROOT / "logs"
CHECKPOINT_FILE = LOG_DIR / "pipeline_checkpoint.json"

for p in [RESULTS_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 2: API Key (Anthropic Claude)
# Load from Colab Secrets, Kaggle Secrets, .env, or environment

def load_api_key():
    # Colab Secrets
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        print("✓ API key loaded from Colab Secrets")
        return
    except: pass
    
    # Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
        print("✓ API key loaded from Kaggle Secrets")
        return
    except: pass
    
    # .env file
    try:
        from dotenv import load_dotenv
        load_dotenv()
        print("✓ .env loaded")
    except: pass
    
    # Check environment
    if os.environ.get("ANTHROPIC_API_KEY", "").startswith("sk-"):
        print("✓ API key found in environment")
    else:
        print("⚠ ANTHROPIC_API_KEY not set — LLM guidance disabled")

load_api_key()

In [ ]:
# Cell 3: Paper Configuration (from repro.yaml)
# Paper-quality parameters — DO NOT CHANGE

PAPER_CONFIG = {
    # Timeouts
    "PYSR_TIMEOUT": 120,
    "METHOD_TIMEOUT": 360,
    
    # PySR parameters
    "N_ITERATIONS": 25,
    "POPULATIONS": 10,
    "PYSR_POPULATION_SIZE": 33,
    "PYSR_PARSIMONY": 0.01,
    "PYSR_MAXSIZE": 30,
    
    # Task counts
    "N_TASKS_DEFI": 74,
    "N_TASKS_INSTABILITY": 70,
    "N_FEYMAN_TASKS": 30,
    "N_NGUYEN_TASKS": 12,
    
    # Splits
    "PCA_TRAIN_FRAC": 0.40,
    "NN_TIME_LIMIT": 120,
    
    # LLM
    "LLM_MODEL": "claude-sonnet-4-20250514",
    "LLM_RETRIES": 3,
    "LLM_K_RUNS": 30,
    
    # Engine
    "ENGINE_NAME": "hybrid_system_v50_2",
}

# Set environment variables
for key, value in PAPER_CONFIG.items():
    os.environ[key] = str(value)

print("=" * 60)
print("PAPER CONFIGURATION LOADED")
print("=" * 60)
for key, value in PAPER_CONFIG.items():
    print(f"  {key}: {value}")
print("=" * 60)

In [ ]:
# Cell 4: Julia/PySR Segfault Prevention
# Critical for preventing torch/juliacall conflicts

os.environ.setdefault("PYTHON_JULIACALL_HANDLE_SIGNALS", "yes")
os.environ.setdefault("JULIA_NUM_THREADS", "1")

print(f"PYTHON_JULIACALL_HANDLE_SIGNALS = {os.environ.get('PYTHON_JULIACALL_HANDLE_SIGNALS')}")
print(f"JULIA_NUM_THREADS = {os.environ.get('JULIA_NUM_THREADS')}")

In [ ]:
# Cell 5: Checkpoint Helpers

def load_checkpoint():
    """Load pipeline checkpoint status."""
    if CHECKPOINT_FILE.exists():
        try:
            return json.loads(CHECKPOINT_FILE.read_text())
        except:
            return {}
    return {}

def save_checkpoint(state):
    """Save pipeline checkpoint."""
    CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_FILE.write_text(json.dumps(state, indent=2))

def is_done(step_id):
    """Check if step is marked as passed."""
    return load_checkpoint().get(step_id) == "pass"

def mark_done(step_id):
    """Mark step as passed in checkpoint."""
    state = load_checkpoint()
    state[step_id] = "pass"
    save_checkpoint(state)
    print(f"  ✓ {step_id} marked complete")

def clear_step(step_id):
    """Clear a step from checkpoint (force re-run)."""
    state = load_checkpoint()
    if step_id in state:
        del state[step_id]
        save_checkpoint(state)
        print(f"  ✓ {step_id} cleared from checkpoint")

print("Checkpoint helpers ready")
print(f"  Checkpoint: {CHECKPOINT_FILE}")
print(f"  Completed: {[k for k,v in load_checkpoint().items() if v=='pass']}")

In [ ]:
# Cell 6: Run Command Helper
import subprocess

def run_step(step_id, cmd, env_extra=None):
    """Run a step with checkpoint awareness."""
    if is_done(step_id):
        print(f"↩ Skipping {step_id} (already done)")
        return True
    
    print(f"\n{'='*60}")
    print(f"▶ Running: {step_id}")
    print(f"  Command: {' '.join(cmd)}")
    print(f"{'='*60}")
    
    env = {**os.environ}
    if env_extra:
        env.update(env_extra)
    
    result = subprocess.run(cmd, env=env, cwd=REPO_ROOT)
    
    if result.returncode == 0:
        mark_done(step_id)
        return True
    else:
        print(f"✗ {step_id} failed with code {result.returncode}")
        return False

print("Run helper ready")

## Phase 0: Setup & Verification

In [ ]:
# Cell 7: Install Dependencies
!pip install -q anthropic pysr pint python-dotenv pyyaml scikit-learn numpy pandas jupyter matplotlib
print("✓ Dependencies installed")

In [ ]:
# Cell 8: Verify Source Tree
import pathlib

def check_file(path):
    exists = pathlib.Path(path).exists()
    print(f"  {'✓' if exists else '✗'} {path}")
    return exists

critical_files = [
    "hypatiax/tools/symbolic/symbolic_engine.py",
    "hypatiax/tools/symbolic/hybrid_system_v50_2.py",
    "protocols/universal_protocol.py",
    "core/runners/common.py",
]

print("Critical files:")
all_ok = all(check_file(f) for f in critical_files)
print(f"\n{'✓ All critical files present' if all_ok else '✗ Missing files'}")

In [ ]:
# Cell 9: Apply Patches (FIX-C1, FIX-C2, FIX-T1, FIX-T2, FIX-XR3)
if not is_done("patches-apply"):
    run_step("patches-gen", ["python3", "scripts/patches/generate_patches.py"])
    run_step("patches-apply", ["python3", "scripts/patches/apply_patches.py"])
    run_step("patches-verify", ["python3", "scripts/patches/apply_patches.py", "--verify"])
    mark_done("patches-apply")
else:
    print("✓ Patches already applied")

## Phase 1: Core Experiments

In [ ]:
# Cell 10: Exp 1 — DeFi 74-task benchmark (§10.2–10.4, §10.6)
# Expected: 89.2% R²>0.99 · 0 catastrophic · 1.73× speedup
# Wall time: 2-4 hours

if not is_done("exp1"):
    run_step("exp1", [
        sys.executable,
        "hypatiax/protocols/experiment_protocol_ablation_exp1.py"
    ])
else:
    print("✓ Exp 1 already completed")

In [ ]:
# Cell 11: Exp 1b — Portfolio Variance seed sweep (§10.5)
# Expected: P(H>P) ≈ 0.76 across seeds [42,99,123,777,2024]
# Wall time: 20-40 minutes

if not is_done("exp1b"):
    run_step("exp1b", [
        sys.executable,
        "hypatiax/protocols/experiment_protocol_defi_v3.py"
    ], env_extra={
        "DEFI_TASK_FILTER": "portfolio",
        "DEFI_SEEDS": "42,99,123,777,2024"
    })
else:
    print("✓ Exp 1b already completed")

In [ ]:
# Cell 12: Exp 2 — Feynman 30-equation extrapolation (§10.7) [SLOW]
# Expected: 9/30 (30%) · wall time: 4-8 hours
# ⚠ Run this cell separately on Colab (resume with checkpoint)

if not is_done("exp2"):
    print("⚠ This step takes 4-8 hours. Run on Colab with checkpoint resume.")
    run_step("exp2", [
        sys.executable,
        "hypatiax/protocols/experiment_protocol_feynman_exp2.py"
    ])
else:
    print("✓ Exp 2 already completed")

In [ ]:
# Cell 13: Exp 3 — Nguyen-12 SR suite (§10.8 primary) SEED=42
# Expected: 11/12 H (91.7%) · 10/12 P · MW U=113, p=0.0097
# Wall time: 30-90 minutes

if not is_done("exp3"):
    run_step("exp3", [
        sys.executable, "-m",
        "hypatiax.protocols.experiment_protocol_nguyen12_exp3",
        "--seed", "42"
    ], env_extra={"SKIP_PKG_CHECK": "1"})
else:
    print("✓ Exp 3 already completed")

In [ ]:
# Cell 14: Exp 3b — Nguyen-12 stability check (§10.8) seeds 99,123,777,2024
# Wall time: 30-90 minutes

if not is_done("exp3b"):
    run_step("exp3b", [
        sys.executable, "-m",
        "hypatiax.protocols.experiment_protocol_nguyen12_exp3",
        "--seeds", "99", "123", "777", "2024"
    ], env_extra={"SKIP_PKG_CHECK": "1"})
else:
    print("✓ Exp 3b already completed")

## Phase 2: Supplementary Benchmarks

In [ ]:
# Cell 15: Supp A — Hybrid routing improvements (Fix 1–5b)
# Expected: +6pp Fix1, +5pp Fix2, +1pp Fix3
# Wall time: 30-60 minutes

if not is_done("suppA"):
    run_step("suppA", [
        sys.executable,
        "hypatiax/protocols/experiment_protocol_hybrid_routing.py"
    ], env_extra={
        "SKIP_PERF_ANALYSIS": "1",
        "HYPATIAX_CORE_OPTIONAL": "1"
    })
else:
    print("✓ Supp A already completed")

In [ ]:
# Cell 16: Supp B — Noise & sample-complexity sweep [SLOW]
# Expected: EHD 100% at all σ · plateau ≈ N=500
# Wall time: 6-12 hours

if not is_done("suppB"):
    print("⚠ This step takes 6-12 hours. Run on Colab with checkpoint resume.")
    run_step("suppB", [
        sys.executable,
        "hypatiax/protocols/experiment_protocol_noise_sweep.py"
    ])
else:
    print("✓ Supp B already completed")

In [ ]:
# Cell 17: §10.9 — Instability analysis (K=30) [SLOW]
# Expected: Spearman ρ=−0.70, p<0.001 · 70 tasks
# Wall time: 3-6 hours

if not is_done("instability"):
    print("⚠ This step takes 3-6 hours. Run on Colab with checkpoint resume.")
    run_step("instability", [
        sys.executable,
        "hypatiax/protocols/experiment_protocol_instability_rf02_04.py"
    ], env_extra={"LLM_K_RUNS": "30"})
else:
    print("✓ Instability already completed")

In [ ]:
# Cell 18: Extrapolation comparative (§10.8)
# Wall time: 20-40 minutes

if not is_done("extrap"):
    run_step("extrap", [
        sys.executable,
        "hypatiax/protocols/experiment_protocol_extrapolation_comparative.py"
    ], env_extra={"HYPATIAX_CORE_OPTIONAL": "1"})
else:
    print("✓ Extrapolation already completed")

## Phase 3: Verification & Results

In [ ]:
# Cell 19: Verify Results Against Paper Targets
run_step("verify", [
    sys.executable,
    "scripts/patches/verify_results.py",
    "--report"
], env_extra={
    "VERIFY_RESULTS_DIR": str(RESULTS_DIR)
})

In [ ]:
# Cell 20: Generate Figures and Tables

# Figures
run_step("figures", [
    sys.executable,
    "figures/generate_figures.py",
    "--outdir", str(RESULTS_DIR / "figures")
])

# Tables
run_step("tables", [
    sys.executable,
    "scripts/patches/generate_tables.py",
    "--outdir", str(RESULTS_DIR / "tables")
], env_extra={
    "TABLE_OUTDIR": str(RESULTS_DIR / "tables"),
    "VERIFY_RESULTS_DIR": str(RESULTS_DIR)
})

In [ ]:
# Cell 21: Summary Report

def print_summary():
    cp = load_checkpoint()
    passed = [k for k, v in cp.items() if v == "pass"]
    failed = [k for k, v in cp.items() if v == "fail"]
    
    print("=" * 60)
    print("PIPELINE SUMMARY")
    print("=" * 60)
    print(f"\n✓ Completed: {len(passed)} steps")
    for s in passed:
        print(f"    ✓ {s}")
    
    if failed:
        print(f"\n✗ Failed: {len(failed)} steps")
        for s in failed:
            print(f"    ✗ {s}")
    
    # Count results
    if RESULTS_DIR.exists():
        json_files = list(RESULTS_DIR.rglob("*.json"))
        fig_dir = RESULTS_DIR / "figures"
        tbl_dir = RESULTS_DIR / "tables"
        
        print(f"\n📊 Results:")
        print(f"    JSON files: {len(json_files)}")
        if fig_dir.exists():
            print(f"    Figures: {len(list(fig_dir.glob('*.pdf')))} PDFs")
        if tbl_dir.exists():
            print(f"    Tables: {len(list(tbl_dir.glob('*.tex')))} TeX files")
    
    print("\n" + "=" * 60)

print_summary()

In [ ]:
# Cell 22: Download Results (Colab/Kaggle)

try:
    from IPython.display import FileLink, display
    
    # Zip results
    !zip -r hypatiax_results.zip hypatiax/data/results/ logs/ -x "*.pyc" "__pycache__/*" > /dev/null 2>&1
    
    print("📦 Results packaged: hypatiax_results.zip")
    display(FileLink("hypatiax_results.zip"))
    print("\n👉 Right-click → Save link as...")
except Exception as e:
    print(f"Download not available: {e}")
    print("Results are in ./hypatiax/data/results/")